# PV Fault Classification from Infrared Images

This project evaluates validation-selected logit adjustment for a fixed ImageNet-pretrained ResNet-18 classifier using the public InfraredSolarModules dataset.

Dataset: https://github.com/RaptorMaps/InfraredSolarModules

The study addresses the following research questions:

1. Does validation-selected logit adjustment improve macro-F1, balanced accuracy, and recall for rare PV fault classes?
2. What effect does the adjustment have on overall classification accuracy?
3. Which PV fault classes benefit from the adjustment, and which classes remain difficult to classify?

The analysis uses one documented data split and one trained ResNet-18 checkpoint. Results are presented through quantitative metrics, class-level comparisons, error analysis, and a qualitative model-interpretation example.

## Setup

The required libraries, random seed, paths, device, and experiment settings are defined before the analysis.

In [1]:
import importlib.util
import subprocess
import sys

required = {
    "seaborn": "seaborn>=0.13,<1",
    "sklearn": "scikit-learn>=1.4,<2",
    "tqdm": "tqdm>=4.66,<5",
}
missing = [
    package
    for module, package in required.items()
    if importlib.util.find_spec(module) is None
]
if missing:
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        *missing,
    ])

In [2]:
import hashlib
import json
import platform
import random
import sys
import time
import urllib.request
import zipfile
from copy import deepcopy
from importlib.metadata import version as package_version
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from PIL import Image
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
    recall_score,
)
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.models import ResNet18_Weights, resnet18
from tqdm import tqdm

sns.set_theme(style="whitegrid")

In [3]:
SEED = 42
QUICK_RUN = False
IMAGE_SIZE = 128
BATCH_SIZE = 128
EPOCHS = 3 if not QUICK_RUN else 1
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 0
TAU_VALUES = np.arange(0.0, 1.51, 0.1)
BOOTSTRAP_ITERATIONS = 2_000
RARE_CLASSES = {"Diode-Multi", "Hot-Spot", "Hot-Spot-Multi", "Soiling"}


In [ ]:
BASE_DIR = Path("/content") if Path("/content").exists() else Path("/tmp")
WORK_DIR = BASE_DIR / "pv_fault_colab"
DATA_DIR = WORK_DIR / "InfraredSolarModules"
OUTPUT_DIR = WORK_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [4]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


In [5]:
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

print(f"Selected device: {DEVICE}")

Selected device: mps


In [ ]:
set_seed(SEED)
print(f"Random seeds initialized to {SEED}.")

## Data Preparation

The analysis uses the public InfraredSolarModules dataset, which contains infrared images from 12 PV fault classes.

In [6]:
DATASET_URL = (
    "https://raw.githubusercontent.com/RaptorMaps/InfraredSolarModules/"
    "88e2d1febbcefe401c17ec80b8973f36a02a1653/"
    "2020-02-14_InfraredSolarModules.zip"
)
EXPECTED_SHA256 = "b82c706bc719b045ac4f8930570d81767a8a170d0998ca3e09283b585db05b5e"
ARCHIVE_PATH = WORK_DIR / "InfraredSolarModules.zip"
METADATA_PATH = DATA_DIR / "module_metadata.json"


In [7]:
WORK_DIR.mkdir(parents=True, exist_ok=True)
if not ARCHIVE_PATH.exists():
    print("Downloading official InfraredSolarModules archive...")
    urllib.request.urlretrieve(DATASET_URL, ARCHIVE_PATH)

digest = hashlib.sha256(ARCHIVE_PATH.read_bytes()).hexdigest()
if digest != EXPECTED_SHA256:
    raise RuntimeError(f"Dataset checksum mismatch: {digest}")

print("Dataset archive is present and checksum verified.")

In [ ]:
if not METADATA_PATH.exists():
    destination = WORK_DIR.resolve()
    with zipfile.ZipFile(ARCHIVE_PATH) as archive:
        for member in archive.infolist():
            member_path = (WORK_DIR / member.filename).resolve()
            if member_path != destination and destination not in member_path.parents:
                raise RuntimeError(
                    f"Unsafe archive member path: {member.filename}"
                )
        archive.extractall(WORK_DIR)

In [ ]:
image_count = len(list((DATA_DIR / "images").glob("*.jpg")))
print(f"Dataset ready: {image_count:,} images.")